# `lidar-rs` Interactive Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/lidar-rs/blob/main/examples/lidar_rs_demo.ipynb)
[![PyPI](https://img.shields.io/pypi/v/lidar-rs.svg)](https://pypi.org/project/lidar-rs/)
[![crates.io](https://img.shields.io/crates/v/lidar.svg)](https://crates.io/crates/lidar)

**`lidar-rs`** is a pure-Rust port of [`opengeos/lidar`](https://github.com/opengeos/lidar) for delineating the nested hierarchy of surface depressions and elevated features (mounts) in Digital Elevation Models (DEMs), powered by [`terrio`](https://github.com/opengeos/terrio) and [`richdem`](https://github.com/opengeos/richdem-rs) with **zero GDAL or C/C++ dependencies**.

This notebook demonstrates:
1. Downloading sample DEM and DSM GeoTIFFs from `opengeos/lidar-rs`
2. DEM smoothing filters (`MeanFilter`, `MedianFilter`, `GaussianFilter`)
3. Extracting surface depressions / sinks (`extract_sinks`)
4. Delineating the nested depression hierarchy (Wu et al. 2019, `delineate_depressions`)
5. Delineating elevated features / mounts from a DSM (`delineate_mounts`)
6. Pure-Rust geospatial utilities (`image_to_cog`, `resample`, `reproject_image`)

In [ ]:
%pip install -q lidar-rs matplotlib pandas tifffile

In [ ]:
import logging
import time
import urllib.request
from pathlib import Path

import lidar_rs as lidar
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile

logging.getLogger("tifffile").setLevel(logging.ERROR)
print(f"lidar-rs version: {lidar.__version__}")

## 1. Download Sample DEM & DSM Data

In [ ]:
base_url = "https://raw.githubusercontent.com/opengeos/lidar-rs/main/test_data"
data_dir = Path("data")
out_dir = Path("output")
data_dir.mkdir(exist_ok=True)
out_dir.mkdir(exist_ok=True)

for name in ("dem.tif", "dsm.tif"):
    dst = data_dir / name
    if not dst.exists():
        urllib.request.urlretrieve(f"{base_url}/{name}", dst)
    arr = tifffile.imread(dst)
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}, range=[{np.nanmin(arr):.2f}, {np.nanmax(arr):.2f}]")

## 2. DEM Smoothing Filters

In [ ]:
dem_path = data_dir / "dem.tif"
med_path = out_dir / "median.tif"

t0 = time.perf_counter()
lidar.MedianFilter(dem_path, kernel_size=3, out_file=med_path)
t1 = time.perf_counter()

dem_arr = tifffile.imread(dem_path)
mean_arr = lidar.MeanFilter(dem_arr, kernel_size=5)
gauss_arr = lidar.GaussianFilter(dem_arr, sigma=2.0)
print(f"MedianFilter (3x3, file I/O) completed in {(t1 - t0) * 1000:.2f} ms")
print(f"In-memory MeanFilter shape={mean_arr.shape}, GaussianFilter shape={gauss_arr.shape}")

## 3. Extract Surface Depressions (Sinks)

In [ ]:
t0 = time.perf_counter()
sinks = lidar.extract_sinks(med_path, min_size=1000, out_dir=out_dir)
t1 = time.perf_counter()

print(f"extract_sinks completed in {(t1 - t0) * 1000:.2f} ms")
print(f"Found {len(sinks['depressions'])} sink region(s)")
print(f"GeoPackage output: {sinks['gpkg_path']}")
pd.DataFrame(sinks["depressions"])

## 4. Delineate Nested Depression Hierarchy (Wu et al. 2019)

In [ ]:
t0 = time.perf_counter()
deps = lidar.delineate_depressions(
    sinks["sink_path"],
    min_size=1000,
    min_depth=0.3,
    interval=0.3,
    out_dir=out_dir,
    bool_level_shp=True,
)
t1 = time.perf_counter()

print(
    f"delineate_depressions completed in {(t1 - t0) * 1000:.2f} ms: "
    f"{len(deps['depressions'])} nested depressions across {deps['num_regions']} region(s) "
    f"(max level = {deps['max_level']})"
)
print(f"Exported {len(deps['level_shapefiles'])} level shapefile(s)")
pd.DataFrame(deps["depressions"]).head(10)

In [ ]:
depth_arr = tifffile.imread(sinks["depth_path"])
dep_id_arr = tifffile.imread(deps["depression_id_path"])
dep_level_arr = tifffile.imread(deps["depression_level_path"])

fig, axes = plt.subplots(2, 2, figsize=(10, 9))

im0 = axes[0, 0].imshow(dem_arr, cmap="terrain")
axes[0, 0].set_title("Original DEM (m)")
fig.colorbar(im0, ax=axes[0, 0], shrink=0.8)

im1 = axes[0, 1].imshow(np.ma.masked_equal(depth_arr, 0), cmap="Blues")
axes[0, 1].set_title("Sink Depth (m)")
fig.colorbar(im1, ax=axes[0, 1], shrink=0.8)

im2 = axes[1, 0].imshow(np.ma.masked_equal(dep_id_arr, 0), cmap="tab20")
axes[1, 0].set_title("Nested Depression IDs")
fig.colorbar(im2, ax=axes[1, 0], shrink=0.8)

im3 = axes[1, 1].imshow(np.ma.masked_equal(dep_level_arr, 0), cmap="viridis")
axes[1, 1].set_title("Depression Hierarchy Levels")
fig.colorbar(im3, ax=axes[1, 1], shrink=0.8)

for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Delineate Elevated Features (Mounts / Tree Canopies) from a DSM

In [ ]:
dsm_path = data_dir / "dsm.tif"
mounts_dir = out_dir / "mounts"

t0 = time.perf_counter()
mounts = lidar.delineate_mounts(
    dsm_path,
    min_size=1000,
    min_height=0.3,
    interval=0.3,
    out_dir=mounts_dir,
)
t1 = time.perf_counter()

print(
    f"delineate_mounts completed in {(t1 - t0) * 1000:.2f} ms: "
    f"{len(mounts['depressions'])} mounts across {mounts['num_regions']} region(s) "
    f"(max level = {mounts['max_level']})"
)

dsm_arr = tifffile.imread(dsm_path)
mount_level_arr = tifffile.imread(mounts["depression_level_path"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
im0 = axes[0].imshow(dsm_arr, cmap="terrain")
axes[0].set_title("Digital Surface Model (DSM)")
fig.colorbar(im0, ax=axes[0], shrink=0.85)

im1 = axes[1].imshow(np.ma.masked_equal(mount_level_arr, 0), cmap="plasma")
axes[1].set_title("Mount Hierarchy Levels")
fig.colorbar(im1, ax=axes[1], shrink=0.85)

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Pure-Rust Geospatial Utilities (Zero GDAL)

In [ ]:
cog_path = lidar.image_to_cog(dem_path, out_dir / "dem_cog.tif")
resampled_path = out_dir / "dem_2m.tif"
lidar.resample(dem_path, resampled_path, resolution=2.0, resampling="bilinear")
reproj_path = out_dir / "dem_wgs84.tif"
lidar.reproject_image(dem_path, reproj_path, dst_crs="EPSG:4326", resampling="bilinear")

print(f"COG: {cog_path} ({Path(cog_path).stat().st_size:,} bytes)")
print(f"Resampled 2m shape: {tifffile.imread(resampled_path).shape}")
print(f"Reprojected EPSG:4326 shape: {tifffile.imread(reproj_path).shape}")